# Diagnóstico Financiero y Económico por Empresa
Puntúa cada empresa comparando sus ratios con los óptimos reales por CNAE del Excel de referencia.

- **Rojo** (0 pts): por debajo del mínimo óptimo
- **Amarillo** (1 pt): dentro del rango óptimo
- **Verde** (2 pts): por encima del máximo óptimo

| Bloque | Ratios | Máx. puntos |
|---|---|---|
| Posición Financiera | Liquidez, P.Ácida, Solvencia, Endeudamiento, Indep. Financiera, FM vs NOF, Tesorería Inmediata | 14 |
| Posición Económica | Deuda/EBITDA, ROE, ROA, Tangibilidad, Intangibilidad, DSCR | 12 |

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
empresas = pd.read_csv("Data/ratios.csv")

rangos = pd.read_excel("Data/ratios_cnae_diagnostico (RETO).xlsx",sheet_name="Ratios CNAE completo"
)

In [ ]:
#Preparar tabla de rangos convertir CNAE a int y limpiar filas de sección
rangos = rangos[pd.to_numeric(rangos["CNAE"], errors="coerce").notna()].copy()
rangos["CNAE"] = rangos["CNAE"].astype(int)

# Calcular FM vs NOF
empresas["FM_vs_NOF"] = empresas["FM"] - empresas["NOF"]

In [ ]:
#Función para parsear rangos del Excel
def parse_rango(rango):

    if pd.isna(rango) or str(rango).strip() in ['—', '-', '']:
        return None, None
    rango = str(rango).strip()
    rango = rango.replace('%', '').replace('x', '').strip()
    rango = rango.replace(',', '.').replace('–', '-')
    parts = rango.split('-')
    if len(parts) != 2:
        return None, None
    try:
        return float(parts[0].strip()), float(parts[1].strip())
    except:
        return None, None

In [ ]:
# Función de clasificación
def clasificar(valor, minimo, maximo, menor_mejor=False):

    if pd.isna(valor) or minimo is None:
        return None

    if not menor_mejor:
        if valor > maximo:
            return 2   
        elif valor >= minimo:
            return 1   
        else:
            return 0  
    else:
        if valor < minimo:
            return 2  
        elif valor <= maximo:
            return 1  
        else:
            return 0  

In [ ]:
#Definición de ratios por bloque

ratios_financiero = {
    "Liquidez_pts":       ("RATIO DE LIQUIDEZ",              "Liquidez\n",       False),
    "P_Acida_pts":        ("RATIO DE PRUEBA ACIDA",          "P.Ácida\n",        False),
    "Solvencia_pts":      ("RATIO DE SOLVENCIA",             "Solvencia\n",      False),
    "Endeudamiento_pts":  ("RATIO DE ENDEUDAMIENTO",         "Endeud.\n",        True),   # menor = mejor
    "Indep_Fin_pts":      ("RATIO DE INDEPENDENCIA",         "Indep.Fin.\n",     False),
    "FM_vs_NOF_pts":      ("FM_vs_NOF",                      None,               False),  # lógica especial
    "Tesoreria_Inm_pts":  ("RATIO DE TESORERIA INMEDIATA",   "Tesorería\nInm.\n",False),
}

ratios_economico = {
    "Deuda_EBITDA_pts":   ("RATIO DE ENDEUDAMIENTO DEL EBITDA", "Deuda/EBITDA\n", True),  # menor = mejor
    "ROE_pts":            ("ROE",                               "ROE\n",           False),
    "ROA_pts":            ("ROA",                               "ROA\n",           False),
    "Tangibilidad_pts":   ("RATIO DE TANGIBILIDAD",             "Tangib.\n",       False),
    "Intangibilidad_pts": ("RATIO DE INTANGIBILIDAD",           "Intangib.\n",     True),  # menor = mejor
    "DSCR_pts":           ("DSRC (Deuda Servicio Ratio de Cobertura)", "DSCR\n",  False),
}

In [ ]:
# Calcular puntos por empresa
def clasificar_fm_nof(fm, nof):
    if pd.isna(fm) or pd.isna(nof):
        return None
    diff = fm - nof
    if nof == 0:
        return 2 if fm >= 0 else 0
    ratio = diff / abs(nof)
    if ratio > 0.10:
        return 2   # Verde
    elif ratio >= -0.10:
        return 1   # Amarillo
    else:
        return 0   # Rojo

def calcular_puntos(df, ratios_dict, rangos_df):
    df = df.copy()
    cols_pts = list(ratios_dict.keys())

    # 1. Pre-convertir a numérico TODA la columna fuera del bucle
    for col_pt, (col_empresa, col_rango, menor_mejor) in ratios_dict.items():
        if col_empresa in df.columns:
            df[col_empresa] = pd.to_numeric(df[col_empresa], errors="coerce")

    # 2. Ejecutar la lógica fila a fila solo sobre los valores ya numéricos
    for col_pt, (col_empresa, col_rango, menor_mejor) in ratios_dict.items():
        resultados = []
        for _, emp in df.iterrows():
            if col_rango is None:
                pts = clasificar_fm_nof(emp["FM"], emp["NOF"])
                resultados.append(pts)
                continue

            cnae = emp["Código primario CNAE 2009"]
            fila = rangos_df[rangos_df["CNAE"] == cnae]

            if fila.empty:
                resultados.append(None)
                continue

            minimo, maximo = parse_rango(fila.iloc[0][col_rango])
            
            valor = emp[col_empresa]
            resultados.append(clasificar(valor, minimo, maximo, menor_mejor))

        df[col_pt] = resultados

    return df, cols_pts

empresas, pts_fin = calcular_puntos(empresas, ratios_financiero, rangos)
empresas, pts_eco = calcular_puntos(empresas, ratios_economico, rangos)

In [ ]:
#Totales y diagnóstico
empresas["Puntos_Financiero"] = empresas[pts_fin].sum(axis=1, min_count=1)  # máx 14
empresas["Puntos_Economico"]  = empresas[pts_eco].sum(axis=1, min_count=1)  # máx 12


def diag_financiero(pts):
    if pd.isna(pts): return None
    if pts >= 11: return "Fuerte"
    if pts >= 6:  return "Moderada"
    return "Débil"

def diag_economico(pts):
    if pd.isna(pts): return None
    if pts >= 9:  return "Fuerte"
    if pts >= 5:  return "Moderada"
    return "Débil"


empresas["Diag_Financiero"] = empresas["Puntos_Financiero"].apply(diag_financiero)
empresas["Diag_Economico"]  = empresas["Puntos_Economico"].apply(diag_economico)

In [ ]:
# Resumen 
cols_ver = (
    ["Nombre", "Código primario CNAE 2009"] +
    pts_fin + ["Puntos_Financiero", "Diag_Financiero"] +
    pts_eco  + ["Puntos_Economico",  "Diag_Economico"]
)

print(empresas[cols_ver].head(10).to_string())
print("\n── Distribución Posición Financiera ──")
print(empresas["Diag_Financiero"].value_counts())
print("\n── Distribución Posición Económica ──")
print(empresas["Diag_Economico"].value_counts())

In [ ]:
#Z' = 0,717·X₁ + 0,847·X₂ + 3,107·X₃ + 0,420·X₄ + 0,998·X₅
#X₁ = Fondo de Maniobra / Activo Total
X1 = (empresas["Activo circulante\nmil EUR\nÚlt. año disp."]- empresas["Pasivo líquido\nmil EUR\nÚlt. año disp."]) / empresas['Total activo\nmil EUR\nÚlt. año disp.']
#X₂ = reserbas / Activo Total
X2 = (empresas["Fondos propios\nmil EUR\nÚlt. año disp."]-empresas["Capital suscrito\nmil EUR\nÚlt. año disp."]) / empresas['Total activo\nmil EUR\nÚlt. año disp.']
#X₃ = EBIT / Activo Total
X3 = empresas["Resultado Explotación\nmil EUR\nÚlt. año disp."] / empresas['Total activo\nmil EUR\nÚlt. año disp.']
#X₄ = Patrimonio neto / Pasivo Total
X4 = empresas["Fondos propios\nmil EUR\nÚlt. año disp."]/ (empresas["Pasivo líquido\nmil EUR\nÚlt. año disp."]+ empresas["Pasivo fijo\nmil EUR\nÚlt. año disp."])
# X5 = Ventas / activo total
X5 = empresas["Importe neto Cifra de Ventas\nmil EUR\nÚlt. año disp."] / empresas['Total activo\nmil EUR\nÚlt. año disp.']

empresas["Altman Z_score"]= +((0.717 * X1) + (0.847 * X2) +(3.107 * X3) + (0.420* X4) + (0.998 * X5))

In [ ]:
def clasificacion_altman(z):
    if z < 1.23:
        return 'Zona de peligro'
    elif z < 2.99:
        return 'Zona gris'
    else:
        return 'Zona segura'

empresas['altman_categoria'] = empresas['Altman Z_score'].apply(clasificacion_altman)

In [ ]:
empresas

In [ ]:
#columna altman_categoria con los valores 'Zona de peligro', 'Zona gris' y 'Zona segura' cuantos hay en cada categoría
print(empresas['altman_categoria'].value_counts())


In [ ]:
print(empresas['Diag_Financiero'].value_counts())
print(empresas['Diag_Economico'].value_counts())

In [ ]:
empresas_filtrado = empresas[["Provincia","altman_categoria","Diag_Financiero","Diag_Economico"]]
empresas_filtrado = empresas_filtrado[empresas_filtrado['altman_categoria'] == "Zona de peligro"]

In [ ]:
print(empresas['Provincia'].value_counts())
print(empresas_filtrado['Provincia'].value_counts())

In [ ]:
# Porcentaje dinámico de empresas en zona de peligro por provincia
total_por_provincia = empresas['Provincia'].value_counts()
peligro_por_provincia = empresas_filtrado['Provincia'].value_counts()

# Pandas alinea los índices (provincias) automáticamente al dividir
porcentaje_peligro = (peligro_por_provincia / total_por_provincia * 100).dropna()

# Ordenar de mayor a menor riesgo
porcentaje_peligro = porcentaje_peligro.sort_values(ascending=False)

print("Porcentaje de empresas en 'Zona de Peligro' por provincia:")
print(porcentaje_peligro.round(2).astype(str) + " %")

In [ ]:

print(empresas_filtrado['Diag_Financiero'].value_counts())
print(empresas_filtrado['Diag_Economico'].value_counts())